In [4]:
import re
import pandas as pd
from clean_recipes import load_recipes, handle_missing, clean_text, enforce_id_integrity

df = load_recipes("recipes.csv")
df = handle_missing(df)
df = clean_text(df)
df = enforce_id_integrity(df)

Initial shape: (522517, 4)

<class 'pandas.DataFrame'>
RangeIndex: 522517 entries, 0 to 522516
Data columns (total 4 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   RecipeId               522517 non-null  int64
 1   Name                   522517 non-null  str  
 2   RecipeCategory         521766 non-null  str  
 3   RecipeIngredientParts  522517 non-null  str  
dtypes: int64(1), str(3)
memory usage: 15.9 MB

No missing RecipeId values.
No duplicate RecipeId values.


In [5]:
df = df[df["Name"].str.len() >= 3]
df = df.drop_duplicates(subset=["Name", "RecipeIngredientParts"], keep="first")
df["Name"] = df["Name"].str.replace("&amp;", "&", regex=False)
df = df.reset_index(drop=True)
print(df.shape)

(520377, 4)


In [6]:
kw = pd.read_csv("recipes.csv", usecols=["RecipeId", "Keywords"])
df = df.merge(kw, on="RecipeId", how="left")
df["Keywords"] = df["Keywords"].fillna("")

def parse_r_list(s):
    return re.findall(r'"([^"]*)"', s)

df["kw_list"] = df["Keywords"].apply(parse_r_list)
df["ing_list"] = df["RecipeIngredientParts"].apply(parse_r_list)
df["ingredients_text"] = df["ing_list"].apply(", ".join)
df[["Name", "kw_list", "ing_list"]].head()

,Name,kw_list,ing_list
0,Low-Fat Berry Blue Frozen Dessert,"[Dessert, Low Protein, Low Cholesterol, Health...","[blueberries, granulated sugar, vanilla yogurt..."
1,Biryani,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[saffron, milk, hot green chili peppers, onion..."
2,Best Lemonade,"[Low Protein, Low Cholesterol, Healthy, Summer...","[sugar, lemons, rind of, lemon, zest of, fresh..."
3,Carina's Tofu-Vegetable Kebabs,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[extra firm tofu, eggplant, zucchini, mushroom..."
4,Cabbage Soup,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[plain tomato juice, cabbage, onion, carrots, ..."


In [7]:
print(df["ing_list"][2])

['sugar', 'lemons, rind of', 'lemon, zest of', 'fresh water', 'fresh lemon juice']


In [8]:
(df["ing_list"].str.len() == 0).sum()

np.int64(2189)

In [9]:
clear = {
 "dessert": ["Dessert","Pie","Bar Cookie","Drop Cookies","Candy","Frozen Desserts","Cheesecake",
             "Tarts","Gelatin","Ice Cream","Chocolate Chip Cookies","Lemon Cake","Peanut Butter Pie",
             "Bread Pudding","Apple Pie","Key Lime Pie","Coconut Cream Pie","Desserts Fruit"],
 "beverage": ["Beverages","Punch Beverage","Smoothies","Shakes","Margarita"],
 "soup": ["Chowders","Clear Soup","Stew","Gumbo","Bean Soup","Ham And Bean Soup","Black Bean Soup",
          "Potato Soup","Broccoli Soup","Mushroom Soup","Soups Crock Pot","Stocks"],
 "sauce/condiment": ["Sauces","Salad Dressings","Spreads","Chutneys","Jellies","Spaghetti Sauce",
                     "Turkey Gravy","Summer Dip"],
 "bread": ["Breads","Quick Breads","Yeast Breads","Scones","Sourdough Breads","Buttermilk Biscuits","Wheat Bread"],
 "breakfast": ["Breakfast","Brunch","Oatmeal","Breakfast Eggs","Breakfast Casseroles"],
 "appetizer/snack": ["Gluten Free Appetizers"],
 "main course": ["Savory Pies","Curries","Meatloaf","Meatballs","Pot Pie","Pot Roast","Stir Fry",
                 "Main Dish Casseroles","Macaroni And Cheese","Spaghetti","Penne","Pasta Shells",
                 "Manicotti","Steak","Roast Beef"],
 "non-food": ["Bath/Beauty","Household Cleaner","Homeopathy/Remedies"],
}

likely = {
 "main course": ["One Dish Meal","Chicken","Meat","Pork","Chicken Breast","Poultry","Lamb/Sheep","Ham",
                 "Whole Chicken","Chicken Thigh & Leg","Tuna","Veal","Halibut","Tilapia","Turkey Breasts",
                 "Deer","Mussels","Whole Turkey","Lobster","Catfish","Crab","Trout","Wild Game",
                 "Beef Organ Meats","Crawfish","Squid","Mahi Mahi","Duck","Chicken Livers","Duck Breasts",
                 "Orange Roughy","Whole Duck","Rabbit","Whitefish","Quail","Pheasant","Goose","Octopus",
                 "Elk","Moose","Bear","Beef Liver","Roast","Bass","Perch","Oysters","Fish Tuna",
                 "Fish Salmon","Roast Beef Crock Pot","Chicken Crock Pot"],
 "side dish": ["Vegetable","Potato","Beans","Onions","Rice","Grains","Corn","Yam/Sweet Potato","Spinach",
               "Greens","Peppers","Cauliflower","Black Beans","Lentil","Long Grain Rice","White Rice",
               "Brown Rice","Short Grain Rice","Medium Grain Rice","Mashed Potatoes","Collard Greens",
               "Chard","Artichoke","Soy/Tofu","Tempeh"],
}

In [10]:
clear_map  = {cat: t for t, cats in clear.items()  for cat in cats}
likely_map = {cat: t for t, cats in likely.items() for cat in cats}

print(len(clear_map), len(likely_map))

74 75


In [11]:
all_cats = set(df["RecipeCategory"])
print([c for c in list(clear_map) + list(likely_map) if c not in all_cats])

[]


In [12]:
df["dish_type"] = df["RecipeCategory"].map(clear_map)
df["confidence"] = "high"
df["source"] = "category"

In [13]:
unresolved = df["dish_type"].isna()
df.loc[unresolved, "dish_type"] = df.loc[unresolved, "RecipeCategory"].map(likely_map)

likely_rows = unresolved & df["dish_type"].notna()
df.loc[likely_rows, "confidence"] = "medium"

In [14]:
still_empty = df["dish_type"].isna()
df.loc[still_empty, "confidence"] = "needs review"
df.loc[still_empty, "source"] = "none"

In [15]:
print(df["confidence"].value_counts())
print(df["dish_type"].value_counts(dropna=False))

confidence
high            224745
medium          171067
needs review    124565
Name: count, dtype: int64
dish_type
NaN                124565
main course        116940
dessert            100082
side dish           69672
bread               29985
sauce/condiment     22738
beverage            22681
breakfast           21818
soup                10866
non-food             1029
appetizer/snack         1
Name: count, dtype: int64


In [16]:
df[df["confidence"] == "needs review"]["RecipeCategory"].value_counts().head(15)

RecipeCategory
Lunch/Snacks       32512
< 60 Mins           9653
< 30 Mins           8964
Cheese              8437
< 15 Mins           6607
Low Protein         6601
< 4 Hours           4938
Low Cholesterol     3783
European            3676
Very Low Carbs      3355
Fruit               2560
Asian               2189
Mexican             2110
Healthy             1710
Weeknight           1334
Name: count, dtype: int64

In [17]:
rules = [
    # exceptions first: these contain "cake" or "roll" but are not desserts or breads
    ("main course",     r"\b(?:crab|fish|salmon|tuna|potato|rice|shrimp|chicken|veggie|vegetable) cakes?\b|\b(?:egg|spring) rolls?\b"),
    ("soup",            r"\b(?:soups?|chowder|stew|bisque|broth)\b"),
    ("salad",           r"\bsalads?\b"),
    ("main course",     r"\b(?:pot pie|shepherd|casserole|lasagna|chili|curry|stir fry|burgers?|tacos?|pizza|sandwich(?:es)?)\b"),
    ("dessert",         r"\b(?:desserts?|cakes?|cookies?|brownies|pudding|cobbler|fudge|ice cream|cupcakes?|pies?|tarts?|mousse|custard)\b"),
    ("beverage",        r"\b(?:beverages?|smoothies?|lemonade|cocktail|punch|milkshake|tea|coffee|margarita)\b"),
    ("sauce/condiment", r"\b(?:sauces?|dressing|dips?|salsa|gravy|marinade|spread|relish|chutney)\b"),
    ("bread",           r"\b(?:bread|rolls?|biscuits?|muffins?|scones?|bagels?|focaccia)\b"),
    ("breakfast",       r"\b(?:pancakes?|waffles?|omelet|omelette|french toast|granola|breakfast)\b"),
    ("appetizer/snack", r"\b(?:appetizers?|snacks?|finger food)\b"),
]

def match_rules(text):
    text = text.str.lower()
    result = pd.Series("", index=text.index, dtype="object")
    for dish, pattern in rules:
        hit = (result == "") & text.str.contains(pattern, regex=True)
        result.loc[hit] = dish
    return result

In [18]:
test = pd.Series(["Cabbage Soup", "Crab Cakes", "Banana Bread", "Chocolate Lava Cake", "Greek Salad", "Mystery Dish"])
print(match_rules(test).tolist())

['soup', 'main course', 'bread', 'dessert', 'salad', '']


In [19]:
name_type = match_rules(df["Name"])
kw_type = match_rules(df["kw_list"].apply(" | ".join))
med = df["confidence"] == "medium"

print((med & (name_type != "")).sum(), "medium rows where the name gives a signal")
print((med & (kw_type != "")).sum(), "medium rows where the keywords give a signal")
print((med & (name_type == "") & (kw_type != "")).sum(), "medium rows where only the keywords give a signal")

62608 medium rows where the name gives a signal
4718 medium rows where the keywords give a signal
3162 medium rows where only the keywords give a signal


In [20]:
name_type = match_rules(df["Name"])

# a specific word in the name overrides a "likely" category (Cabbage Soup under Vegetable)
override = (df["confidence"] == "medium") & (name_type != "") & (name_type != "main course")
df.loc[override, "dish_type"] = name_type[override]
df.loc[override, "source"] = "name (override)"

# unresolved recipes: try the name first
by_name = df["dish_type"].isna() & (name_type != "")
df.loc[by_name, "dish_type"] = name_type[by_name]
df.loc[by_name, "confidence"] = "medium"
df.loc[by_name, "source"] = "name"

# then the Keywords tags
kw_type = match_rules(df["kw_list"].apply(" | ".join))
by_kw = df["dish_type"].isna() & (kw_type != "")
df.loc[by_kw, "dish_type"] = kw_type[by_kw]
df.loc[by_kw, "confidence"] = "low"
df.loc[by_kw, "source"] = "keywords"

# nothing matched
df["dish_type"] = df["dish_type"].fillna("unknown")

In [21]:
df["conflict"] = (df["source"] == "category") & (name_type != "") & (name_type != df["dish_type"])

In [22]:
print(df["confidence"].value_counts())
print(df["dish_type"].value_counts())
print(df["conflict"].sum(), "conflicts")

confidence
medium          228497
high            224745
needs review     66692
low                443
Name: count, dtype: int64
dish_type
main course        114706
dessert            105280
unknown             66692
sauce/condiment     47216
side dish           46823
bread               33659
soup                31776
salad               25629
beverage            23612
breakfast           22726
appetizer/snack      1229
non-food             1029
Name: count, dtype: int64
29390 conflicts


In [23]:
review = df[df["confidence"] == "needs review"]
print(review["RecipeCategory"].value_counts().head(20))
review[["Name", "RecipeCategory"]].sample(20, random_state=1)

RecipeCategory
Lunch/Snacks       17020
< 60 Mins           4967
< 30 Mins           4780
Cheese              4619
Low Protein         3159
< 15 Mins           3136
Very Low Carbs      2438
European            2298
< 4 Hours           2290
Low Cholesterol     1829
Fruit               1319
Asian               1310
Healthy             1154
Mexican              996
High Protein         913
Weeknight            696
Apple                669
Kid Friendly         536
Vegan                463
Pineapple            447
Name: count, dtype: int64


,Name,RecipeCategory
456773,Grilled Tilapia With Chimichurri Mayo,South American
441628,Deviled Almonds,Very Low Carbs
411083,Greek Seafood Feast for the Eyes and Palate,Very Low Carbs
499582,Chipotle Deviled Eggs and Bacon,< 60 Mins
412892,Whole Pears Roasted in Garlic,< 60 Mins
483075,Peanut Butter Energy Bites,< 15 Mins
4252,Barbecue Ribs,Weeknight
308464,Cinnamon Sug' Sweet Potato Sticks,Low Protein
483826,Kool-Aid Popcorn,< 60 Mins
235641,Basil Butter,Very Low Carbs


In [24]:
name_type = match_rules(df["Name"])
conf = df[df["conflict"]].copy()
conf["name_type"] = name_type[conf.index]

print(conf.groupby(["dish_type", "name_type"]).size().sort_values(ascending=False).head(12))
conf[["Name", "RecipeCategory", "dish_type", "name_type"]].sample(20, random_state=1)

dish_type        name_type      
side dish        main course        6550
sauce/condiment  salad              2585
bread            dessert            2433
breakfast        main course        1666
                 dessert            1377
main course      sauce/condiment    1368
breakfast        bread              1367
main course      dessert            1188
dessert          salad              1045
                 bread              1008
                 main course         924
                 sauce/condiment     829
dtype: int64


,Name,RecipeCategory,dish_type,name_type
250738,Ya Gotta Try It Breakfast Casserole,Breakfast,breakfast,main course
234730,Key Lime Pie Martini,Beverages,beverage,dessert
416505,&quot;olive Garden&quot; Salad and Dressing Re...,Salad Dressings,sauce/condiment,salad
99732,Old-Fashioned Hot Fudge Sauce,Sauces,sauce/condiment,dessert
457035,Strawberry Jello Salad (Healthy),Gelatin,dessert,salad
243391,Apple Top Coffee Cake,Breads,bread,dessert
279607,Fat Free Sugar Free Blueberry Muffins,Breakfast,breakfast,bread
499969,Curry Dip,Sauces,sauce/condiment,main course
191760,Grandpa Rondina's Meatballs and Sauce,Spaghetti,main course,sauce/condiment
162763,Shrimp Pasta Salad With Dill and Parsley,Pasta Shells,main course,salad


In [25]:
pairs = [("side dish","main course"), ("bread","dessert"), ("breakfast","main course"),
         ("breakfast","dessert"), ("main course","sauce/condiment"), ("breakfast","bread"),
         ("main course","dessert"), ("dessert","salad")]

for a, b in pairs:
    sub = conf[(conf["dish_type"] == a) & (conf["name_type"] == b)]
    print(f"\n{a} (category) vs {b} (name): {len(sub)}")
    print(sub[["Name", "RecipeCategory"]].sample(8, random_state=1).to_string(index=False))


side dish (category) vs main course (name): 6550
                                                   Name RecipeCategory
                            Green Bean & Corn Casserole          Beans
                                  Melanie's Pizza Dough         Grains
                                Cheese - Corn Casserole           Corn
                      Roasted Corn and Black Bean Chili    Black Beans
                                      Tim Hortons Chili         Onions
Falafel W/ Avocado Spread - Veggie Burgers W/ Guacamole          Beans
                                        Baked Egg Rolls      Vegetable
                                       Vegetarian Chili          Beans

bread (category) vs dessert (name): 2433
                                             Name RecipeCategory
Bad for Hips, Good for Lips Blueberry Coffee Cake         Breads
                     Cinnamon Coffee Cake Muffins   Quick Breads
               Delicious Graham Apple Coffee Cake         Breads
         

In [61]:
modifiers = re.compile(r"\b(?:fresh|freshly|chopped|minced|diced|sliced|grated|shredded|crushed|finely|coarsely|thinly|boneless|skinless|unsalted|salted|softened|melted|ripe|packed|organic|large|medium|small)\b")

keep_s = {"molasses", "hummus", "asparagus", "couscous", "swiss", "citrus", "watercress", "grits"}

def singular(word):
    if word in keep_s or len(word) <= 3:
        return word
    if word.endswith("ies") and len(word) > 4:
        return word[:-3] + "y"
    if word.endswith("oes"):
        return word[:-2]
    if word.endswith(("ches", "shes", "sses", "xes")):
        return word[:-2]
    if word.endswith(("ss", "us")):
        return word
    if word.endswith("s"):
        return word[:-1]
    return word

synonyms = {
    "garlic clove": "garlic",
    "all-purpose flour": "flour", "all purpose flour": "flour", "plain flour": "flour",
    "granulated sugar": "sugar", "white sugar": "sugar", "white granulated sugar": "sugar",
    "confectioners sugar": "powdered sugar", "confectioners' sugar": "powdered sugar", "icing sugar": "powdered sugar",
    "vanilla extract": "vanilla", "pure vanilla extract": "vanilla",
    "extra virgin olive oil": "olive oil", "extra-virgin olive oil": "olive oil",
    "kosher salt": "salt", "sea salt": "salt", "table salt": "salt",
    "ground black pepper": "black pepper", "cracked black pepper": "black pepper",
    "ground cinnamon": "cinnamon", "ground nutmeg": "nutmeg", "ground cumin": "cumin",
    "bay leave": "bay leaf",
    "scallion": "green onion", "spring onion": "green onion",
    "yellow onion": "onion", "white onion": "onion",
    "whipping cream": "heavy cream", "heavy whipping cream": "heavy cream",
}

def normalize(orig):
    s = modifiers.sub("", orig.lower().strip())
    s = re.sub(r"\s+", " ", s).strip(" ,-")
    if not s:
        return orig.lower().strip()
    words = s.split(" ")
    words[-1] = singular(words[-1])
    s = " ".join(words)
    return synonyms.get(s, s)

In [62]:
norm_map = {s: normalize(s) for s in counts.index}
canon = pd.Series(norm_map)
print(len(counts), "->", canon.nunique(), "distinct ingredients after normalization")

merged = pd.DataFrame({"original": counts.index, "count": counts.values, "canonical": canon.reindex(counts.index).values})
top = merged.groupby("canonical").agg(total=("count", "sum"), variants=("original", "nunique")).sort_values("total", ascending=False)

for name in top[top["variants"] > 1].head(25).index:
    vs = merged[merged["canonical"] == name].sort_values("count", ascending=False)["original"].head(6).tolist()
    print(name, "<-", vs)

7302 -> 6392 distinct ingredients after normalization
salt <- ['salt', 'kosher salt', 'sea salt', 'table salt']
butter <- ['butter', 'unsalted butter', 'salted butter']
sugar <- ['sugar', 'granulated sugar', 'white sugar']
garlic <- ['garlic cloves', 'garlic', 'garlic clove', 'fresh garlic', 'minced garlic cloves', 'fresh garlic cloves']
egg <- ['eggs', 'egg']
onion <- ['onion', 'onions', 'yellow onion', 'yellow onions', 'small onion', 'fresh onion']
flour <- ['flour', 'all-purpose flour', 'plain flour']
olive oil <- ['olive oil', 'extra virgin olive oil']
water <- ['water', 'fresh water']
vanilla <- ['vanilla', 'vanilla extract', 'pure vanilla extract']
black pepper <- ['black pepper', 'fresh ground black pepper', 'ground black pepper', 'cracked black pepper', 'fresh black pepper']
pepper <- ['pepper', 'fresh pepper']
cinnamon <- ['cinnamon', 'ground cinnamon']
lemon juice <- ['lemon juice', 'fresh lemon juice']
tomato <- ['tomato', 'diced tomatoes', 'crushed tomatoes', 'chopped tomat

In [63]:
def clean_list(lst):
    out = [norm_map.get(s.lower().strip(), s.lower().strip()) for s in lst]
    return list(dict.fromkeys(x for x in out if x))

df["ing_clean"] = df["ing_list"].apply(clean_list)
df[["Name", "ing_list", "ing_clean"]].head()

,Name,ing_list,ing_clean
0,Low-Fat Berry Blue Frozen Dessert,"[blueberries, granulated sugar, vanilla yogurt...","[blueberries, sugar, vanilla yogurt, lemon juice]"
1,Biryani,"[saffron, milk, hot green chili peppers, onion...","[saffron, milk, hot green chili pepper, onion,..."
2,Best Lemonade,"[sugar, lemons, rind of, lemon, zest of, fresh...","[sugar, lemons, rind of, lemon, zest of, water..."
3,Carina's Tofu-Vegetable Kebabs,"[extra firm tofu, eggplant, zucchini, mushroom...","[extra firm tofu, eggplant, zucchini, mushroom..."
4,Cabbage Soup,"[plain tomato juice, cabbage, onion, carrots, ...","[plain tomato juice, cabbage, onion, carrot, c..."


In [64]:
ing = df[["RecipeId", "ing_clean"]].explode("ing_clean").dropna()
ing["ingredient"] = ing["ing_clean"].str.lower().str.strip()
counts = ing["ingredient"].value_counts()

print(len(counts), "distinct ingredient strings")
print(counts.head(60).to_string())

for n in [100, 300, 500, 1000]:
    print(n, "->", round(100 * counts.head(n).sum() / counts.sum(), 1), "% of all ingredient uses")

6479 distinct ingredient strings
ingredient
salt                    212448
butter                  145503
sugar                   123356
garlic                  119710
egg                     117686
onion                   114940
flour                   102579
olive oil                88852
water                    79660
vanilla                  59497
milk                     58561
black pepper             53997
pepper                   48768
cinnamon                 43007
lemon juice              42872
brown sugar              40628
baking powder            39183
green onion              36205
baking soda              32811
carrot                   32198
parsley                  32190
parmesan cheese          32029
sour cream               26804
tomatoes                 26664
margarine                24564
powdered sugar           23773
cream cheese             23368
garlic powder            22890
celery                   22456
cumin                    22267
honey                    2

In [65]:
import re

ing_rules = [
    ("baking & flavoring", r"baking (?:powder|soda)|cream of tartar|\byeast\b|\bextract\b|\bessence\b|^vanilla(?: beans?)?$|\bgelatin\b|\bcocoa\b|\bchocolate\b|food coloring|\bpudding\b"),
    ("liquid & sauce",     r"\b(?:broth|stock|bouillon|sauce|salsa|ketchup|catsup|mustard|mayonnaise|mayo|vinegar|wine|beer|rum|whiskey|bourbon|brandy|vodka|liqueur|sherry|kahlua|marnier|triple sec|mirin|amaretto|schnapps|liquid smoke|miracle whip|juice|water|coffee|tea|soup|gravy|dressing|marinade|cider|coconut milk|coconut cream|cream of coconut|pesto|tahini|miso)\b"),
    ("nuts & legumes",     r"\b(?:peanut|almond|nut) butter\b"),
    ("fat & oil",          r"\b(?:butter|margarine|shortening|lard|oil|ghee|cooking spray)\b"),
    ("spice & herb",       r"\b(?:garlic|onion|celery) (?:powder|salt|seed)\b|\b(?:black|white|ground|cayenne|lemon) pepper\b|ground red pepper|^pepper$|pepper flakes|crushed red pepper|peppercorns?|\b(?:caraway|fennel|anise|cumin|coriander) seeds?\b|\b(?:salt|cinnamon|cumin|paprika|oregano|basil|thyme|parsley|cilantro|dill|rosemary|sage|bay leaf|bay leaves|nutmeg|turmeric|coriander|ginger|gingerroot|allspice|cardamom|mint|chives?|tarragon|marjoram|saffron|seasoning|pie spice|curry powder|chili powder|poppy seeds?|sesame seeds?|mustard seeds?|cayenne|herbs?|garam masala|lemongrass|horseradish)\b|ground cloves?|whole cloves?|^cloves?$"),
    ("grain & starch",     r"\b(?:flour|cornflour|rice|pasta|spaghetti|macaroni|noodles?|penne|fettuccine|lasagna|linguine|bread|breadcrumbs?|bread crumbs?|crackers?|cornmeal|cornstarch|corn starch|oats?|oatmeal|tortillas?|buns?|rolls?|bagels?|couscous|quinoa|barley|cereal|panko|dough|pastry|phyllo|filo|crust|masa|grits|polenta|wheat germ|potato(?:es)?|yams?|stuffing)\b"),
    ("dairy",              r"\b(?:milk|soymilk|buttermilk|cheese|cheddar|mozzarella|parmesan|feta|ricotta|cream|yogurt|yoghurt|half-and-half|half and half|creme fraiche|whipped topping|cool whip|ice cream|custard)\b"),
    ("egg",                r"\beggs?\b"),
    ("seafood",            r"\b(?:shrimp|prawns?|salmon|tuna|crab|crabmeat|crab meat|lobster|fish|cod|tilapia|scallops?|clams?|oysters?|anchov(?:y|ies)|sardines?|halibut|mussels?|catfish|trout|squid|crawfish|calamari)\b"),
    ("meat & poultry",     r"\b(?:chicken|beef|pork|bacon|ham|sausages?|turkey|lamb|veal|steaks?|pepperoni|salami|prosciutto|hot dogs?|bologna|venison|duck|chorizo|meatballs?|brisket|hamburger|bratwurst|kielbasa|pancetta)\b"),
    ("sweetener",          r"\b(?:sugar|honey|syrup|molasses|sweetener|splenda|stevia|agave|marshmallows?|marshmallow creme|jam|jelly|preserves|marmalade|sprinkles|frosting|icing|caramel|candy)\b"),
    ("vegetable",          r"\b(?:onions?|shallots?|scallions?|garlic|tomato(?:es)?|carrots?|celery|spinach|broccoli|cauliflower|cabbage|lettuce|cucumbers?|zucchini|squash|mushrooms?|corn|peppers?|chil(?:ies|i|e)s?|jalape\w+|serrano|habanero|eggplant|asparagus|green beans?|kale|leeks?|beets?|radish(?:es)?|turnips?|artichokes?|okra|avocados?|peas?|olives?|pickles?|capers?|bok choy|arugula|sprouts?|fennel|parsnips?|pumpkin|greens|pimientos?|pimentos?|sauerkraut)\b"),
    ("fruit",              r"\b(?:lemons?|limes?|oranges?|apples?|applesauce|bananas?|\w*berr(?:y|ies)|cherr(?:y|ies)|peach(?:es)?|pears?|pineapple|mangos?|mangoes|grapes?|raisins?|dates?|figs?|coconut|apricots?|plums?|melons?|watermelon|cantaloupe|zest|rind|currants?|prunes?|kiwi|papaya|grapefruit|tangerines?|rhubarb|fruit)\b"),
    ("nuts & legumes",     r"\b(?:peanuts?|almonds?|walnuts?|pecans?|cashews?|pistachios?|hazelnuts?|macadamias?|pine nuts?|nuts?|beans?|lentils?|chickpeas?|garbanzo|tofu|tempeh|hummus|sunflower|flax|chia)\b"),
]

compiled = [(label, re.compile(pat)) for label, pat in ing_rules]

def label_ingredient(s):
    for label, pat in compiled:
        if pat.search(s):
            return label
    return "other"

res = pd.DataFrame({"count": counts})
res["group"] = [label_ingredient(s) for s in res.index]

share = (res.groupby("group")["count"].sum() / res["count"].sum() * 100).round(1)
print(share.sort_values(ascending=False))

group
spice & herb          21.4
vegetable             17.4
liquid & sauce        12.2
dairy                  8.4
fat & oil              7.3
grain & starch         7.0
sweetener              6.6
baking & flavoring     4.4
fruit                  4.2
meat & poultry         3.5
egg                    3.1
nuts & legumes         2.3
other                  1.5
seafood                0.7
Name: count, dtype: float64


In [66]:
res[res["group"] == "other"].head(40)

,count,group
ingredient,,
green chily,1540,other
baguette,945,other
bisquick,943,other
red capsicum,858,other
gorgonzola,840,other
crouton,833,other
wonton wrapper,798,other
watercress,789,other
cognac,786,other


In [67]:
# 1) fix the "chily" artifact left by the plural-to-singular step
df["ing_clean"] = df["ing_clean"].apply(lambda L: [x.replace("chily", "chili") for x in L])

# 2) exact-name groups for the leftovers (add new lines here each round)
manual_groups = {
    "red capsicum": "vegetable", "watercress": "vegetable", "bamboo shoot": "vegetable",
    "jicama": "vegetable", "swiss chard": "vegetable", "rotel": "vegetable",
    "baguette": "grain & starch", "french baguette": "grain & starch", "crouton": "grain & starch",
    "wonton wrapper": "grain & starch", "english muffin": "grain & starch",
    "frozen hash brown": "grain & starch", "bisquick": "grain & starch",
    "bisquick baking mix": "grain & starch", "biscuit mix": "grain & starch", "ladyfinger": "grain & starch",
    "gorgonzola": "dairy", "gruyere": "dairy",
    "cognac": "liquid & sauce", "sake": "liquid & sauce", "tamari": "liquid & sauce",
    "dry vermouth": "liquid & sauce",
    "instant espresso powder": "baking & flavoring", "bicarbonate of soda": "baking & flavoring",
    "xanthan gum": "baking & flavoring", "arrowroot": "baking & flavoring",
    "sultana": "fruit", "craisin": "fruit", "nectarine": "fruit",
    "star anise": "spice & herb", "mixed spice": "spice & herb", "mace": "spice & herb",
    "chinese five spice powder": "spice & herb", "five-spice powder": "spice & herb",
    "curry leave": "spice & herb", "red curry paste": "spice & herb",
    "oleo": "fat & oil",
}

compiled = [(label, re.compile(pat)) for label, pat in ing_rules]

def label_ingredient(s):
    if s in manual_groups:
        return manual_groups[s]
    for label, pat in compiled:
        if pat.search(s):
            return label
    return "other"

# 3) recount and regroup
ing = df[["RecipeId", "ing_clean"]].explode("ing_clean").dropna()
ing["ingredient"] = ing["ing_clean"].str.lower().str.strip()
counts = ing["ingredient"].value_counts()

res = pd.DataFrame({"count": counts})
res["group"] = [label_ingredient(s) for s in res.index]

share = (res.groupby("group")["count"].sum() / res["count"].sum() * 100).round(1)
print(share.sort_values(ascending=False))
print(res[res["group"] == "other"].head(30).to_string())

group
spice & herb          21.5
vegetable             17.5
liquid & sauce        12.2
dairy                  8.4
fat & oil              7.3
grain & starch         7.2
sweetener              6.6
baking & flavoring     4.5
fruit                  4.3
meat & poultry         3.5
egg                    3.1
nuts & legumes         2.3
other                  0.9
seafood                0.7
Name: count, dtype: float64
                            count  group
ingredient                              
curry leaves                  547  other
craisins                      446  other
frozen hash browns            440  other
curry paste                   409  other
baking mix                    391  other
curry                         388  other
blue curacao                  384  other
ground mace                   374  other
creme de cacao                364  other
frozen limeade concentrate    355  other
queso fresco                  347  other
tamarind paste                345  other
bulgur        

In [69]:
manual_groups.update({
    "curry leaves": "spice & herb", "curry paste": "spice & herb", "curry": "spice & herb",
    "ground mace": "spice & herb", "msg": "spice & herb", "fenugreek seeds": "spice & herb",
    "asafoetida powder": "spice & herb", "savory": "spice & herb",
    "frozen hash browns": "grain & starch", "hash brown": "grain & starch", "baking mix": "grain & starch",
    "bulgur": "grain & starch", "matzo meal": "grain & starch", "vital wheat gluten": "grain & starch",
    "semolina": "grain & starch", "quick-cooking tapioca": "grain & starch",
    "craisins": "fruit", "plantain": "fruit",
    "blue curacao": "liquid & sauce", "creme de cacao": "liquid & sauce",
    "frozen limeade concentrate": "liquid & sauce", "tamarind paste": "liquid & sauce",
    "sambal oelek": "liquid & sauce", "chutney": "liquid & sauce", "harissa": "liquid & sauce",
    "queso fresco": "dairy", "mascarpone": "dairy",
    "edamame": "nuts & legumes",
    "courgette": "vegetable", "nori": "vegetable",
})

res["group"] = [label_ingredient(s) for s in res.index]
share = (res.groupby("group")["count"].sum() / res["count"].sum() * 100).round(1)
print(share.sort_values(ascending=False))
print(res[res["group"] == "other"].head(15).to_string())

group
spice & herb          21.6
vegetable             17.5
liquid & sauce        12.3
dairy                  8.4
fat & oil              7.3
grain & starch         7.2
sweetener              6.6
baking & flavoring     4.5
fruit                  4.3
meat & poultry         3.5
egg                    3.1
nuts & legumes         2.3
other                  0.7
seafood                0.7
Name: count, dtype: float64
                      count  group
ingredient                        
tapioca starch          276  other
hominy                  273  other
pomegranate seeds       264  other
chervil                 260  other
radicchio               259  other
pico de gallo           254  other
green capsicum          252  other
red snapper fillet      252  other
thai red curry paste    248  other
vermouth                243  other
escarole                234  other
vanilla flavoring       233  other
rump roast              233  other
aubergine               231  other
green curry paste       231 

In [70]:
out = res.reset_index()
out.columns = ["ingredient", "count", "group"]
out.to_csv("ingredient_groups.csv", index=False)

In [78]:
df["has_ingredients"] = df["ing_list"].str.len() > 0
df["ingredients_clean"] = df["ing_clean"].apply("|".join)

out = df[["RecipeId", "Name", "dish_type", "confidence",
          "has_ingredients", "ingredients_clean"]]
out.to_csv("recipe_labels.csv", index=False)
print(out.shape)

(520377, 6)
